# LLM LoRA Fine-tuning 실습

이 노트북은 HuggingFace Trainer를 사용하여 LLM을 LoRA(Low-Rank Adaptation)로 파인튜닝하는 방법을 학습합니다.

## 학습 목표
1. LoRA weight로 파인튜닝하기
2. Base model과 Fine-tuned model의 성능 비교
3. HuggingFace Trainer 코드 이해
4. LoRA config 이해

## 실습 태스크
**Translation**: 한국어 > 영어 번역 태스크
- 평가: 번역이 얼마나 원본과 표면적으로 유사한지 측정 (BLEU Score)

## 1. 환경 설정 및 라이브러리 임포트


In [ ]:
import os
import json
import yaml
import torch
import numpy as np
import re
import random
from typing import Optional
from tqdm import tqdm

# HuggingFace 관련
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    set_seed
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import matplotlib.pyplot as plt
import seaborn as sns

# BLEU score 계산을 위한 라이브러리 추가
import nltk
nltk.download('punkt_tab')
from nltk.translate.bleu_score import corpus_bleu
from nltk.tokenize import word_tokenize

# GPU 확인
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ GPU를 사용할 수 없습니다. CPU로 실행됩니다 (매우 느릴 수 있습니다).")

ModuleNotFoundError: No module named 'torch'

## 2. 설정 파일 로드

In [ ]:
# config.yaml 파일 로드
with open('config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# 설정 출력
print("=" * 60)
print("학습 설정")
print("=" * 60)
print(f"모델: {config['model']['name_or_path']}")
print(f"LoRA rank (r): {config['lora']['r']}")
print(f"LoRA alpha: {config['lora']['lora_alpha']}")
print(f"Target modules: {config['lora']['target_modules']}")
print(f"학습 에포크: {config['training']['num_train_epochs']}")
print(f"배치 크기: {config['training']['per_device_train_batch_size']}")
print(f"학습률: {config['training']['learning_rate']}")
print(f"최대 시퀀스 길이: {config['data']['max_length']}")
print(f"생성 최대 토큰 수: {config['generation']['max_new_tokens']}") # 추가
print(f"샘플링 사용: {config['generation']['do_sample']}") # 추가
if not config['generation']['do_sample']:
    print(f"  -> num_beams: {config['generation']['num_beams']}") # 추가
    print(f"  -> early_stopping: {config['generation']['early_stopping']}") # 추가
else:
    print(f"  -> Temperature: {config['generation']['temperature']}") # 추가

print("=" * 60)

# 랜덤 시드 설정 (완전한 재현성을 위해 모든 라이브러리에 적용)
sid = config['seed']
print(f"\n랜덤 시드 설정: {sid}")

# PyTorch 시드
torch.manual_seed(sid)
torch.cuda.manual_seed_all(sid)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# NumPy 시드
np.random.seed(sid)

# Python random 시드
random.seed(sid)

# Transformers 시드 (HuggingFace)
set_seed(sid)

print("✅ 모든 시드 설정 완료 (재현성 보장)")

## 3. 데이터 로드 및 전처리

**참고**: 원본 데이터는 `data/raw/translation/` 디렉토리에 미리 다운로드되어 제공됩니다.
JSONL 형식의 파일을 읽어서 전처리합니다.

In [ ]:
# 원본 데이터 디렉토리 확인 (config에서 가져오기)
raw_data_dir = config['data']['raw_data_dir']
processed_dir = config['data']['processed_dir']

# 원본 데이터 파일 경로
train_raw_file = os.path.join('''이 부분을 채워주세요''')
test_raw_file = os.path.join('''이 부분을 채워주세요''')
validation_raw_file = os.path.join('''이 부분을 채워주세요''')

# 디렉토리 생성
os.makedirs(processed_dir, exist_ok=True)

print("=" * 60)
print("원본 데이터 로드 및 전처리")
print("=" * 60)

# JSONL 파일 로드 함수
def load_jsonl(file_path: str) -> list[dict]:
    '''이 부분을 채워주세요'''
    return data

# 원본 데이터 로드
print("\n[1/3] 원본 데이터 로드 중...")
print("원본 데이터 경로:", raw_data_dir)

# Train 데이터 로드
if not os.path.exists(train_raw_file):
    raise FileNotFoundError(f"Train 데이터 파일을 찾을 수 없습니다: {train_raw_file}")

train_raw = load_jsonl(train_raw_file)
print(f"  - Train 데이터: {len(train_raw)}개")


# 데이터 구조 확인 (샘플)
if len(train_raw) > 0:
    sample = train_raw[0]
    print(f"\n  [데이터 구조 확인]")
    print(f"    - Keys: {list(sample.keys())}")
    print(f"    - 한국어 (ko): {sample.get('ko', {})}")
    print(f"    - 영어 (en): {sample.get('en', {})}")

# Test/Validation 데이터 로드 (Validation을 우선 사용)
if os.path.exists(validation_raw_file):
    test_raw = load_jsonl(validation_raw_file)
    print(f"  - Validation 데이터 (test로 사용): {len(test_raw)}개")
    use_validation = True
elif os.path.exists(test_raw_file):
    test_raw = load_jsonl(test_raw_file)
    print(f"  - Test 데이터: {len(test_raw)}개")
    use_validation = False
else:
    raise FileNotFoundError(f"Validation 또는 Test 데이터 파일을 찾을 수 없습니다: {validation_raw_file} 또는 {test_raw_file}")


In [ ]:
def format_for_training(example: dict) -> dict:
    """학습용 형식으로 변환"""
    query = example["ko"]
    response = example["en"]

    prompt = f"다음 한국어를 영어로 번역하세요. 오직 한 문장으로 번역하고, 번역이 끝나면 바로 대화 생성을 종료하세요.\n번역해야 하는 한국어 문장:\n{query}\n번역된 영어문장:"

    return {
        "prompt": prompt,
        "response": response,
    }

In [ ]:
# JSONL 파일로 저장하는 함수
def save_to_jsonl(data, filepath: str):
    """데이터를 JSONL 형식으로 저장"""
    with open(filepath, 'w', encoding='utf-8') as f:
        for item in data:
            json.dump(item, f, ensure_ascii=False)
            f.write('\n')

# 전처리된 데이터 저장
print("\n[2/3] 데이터 전처리 중...")

# Train 데이터 전처리
train_processed = ['''이 부분을 채워주세요''']
train_processed_file = os.path.join('''이 부분을 채워주세요''')
save_to_jsonl('''이 부분을 채워주세요'''))
print(f"  - 전처리된 train 데이터 저장: {train_processed_file} ({len(train_processed)}개)")

# Test 데이터 전처리
test_processed = ['''이 부분을 채워주세요''']
test_processed_file = os.path.join(processed_dir, "test.jsonl")
save_to_jsonl(test_processed, test_processed_file)
print(f"  - 전처리된 test 데이터 저장: {test_processed_file} ({len(test_processed)}개)")
if use_validation:
    print("  ⚠️ Validation 데이터를 test로 사용합니다.")

# 데이터 통계 출력
print("\n[3/3] 데이터 통계:")

# 샘플 데이터 출력
print("\n[샘플 데이터]")
print(f"\n===== Train 샘플: =====")
for i, item in enumerate(train_processed):
    print(f"Prompt:\n{item['prompt']}")
    print(f"Response:\n{item['response']}")
    break

print(f"\n===== Test 샘플: =====")
for i, item in enumerate(test_processed):
    print(f"Prompt:\n{item['prompt']}")
    print(f"Response:\n{item['response']}")
    break

print("\n" + "=" * 60)
print("전처리 완료!")
print("=" * 60)

# HuggingFace Dataset으로 변환 (학습용)
train_dataset = Dataset.from_list(train_processed)
test_dataset = Dataset.from_list(test_processed)

# 정답 정보 저장 (평가용)
test_answers = [item['response'] for item in test_processed]


In [ ]:
# 토크나이저 초기화
tokenizer = AutoTokenizer.from_pretrained(config['model']['name_or_path'])

# 패딩 토큰 설정 (없는 경우)
if tokenizer.pad_token is None:
    # pad token, id를 eos token, id로 설정
    '''이 부분을 채워주세요'''
    '''이 부분을 채워주세요'''

print(f"토크나이저: {config['model']['name_or_path']}")
print(f"Vocab 크기: {len(tokenizer)}")
print(f"패딩 토큰: {tokenizer.pad_token}")

# 토크나이징 함수 (Prompt + Response 형식)
def tokenize_function(examples):
    """데이터를 토크나이징 (Prompt + Response 형식, Response만 학습)"""
    prompts = examples[config['data']['prompt_column']]
    responses = examples[config['data']['response_column']]

    # Prompt와 Response를 결합
    texts = [f"{prompt}\n{response}" for prompt, response in zip(prompts, responses)]

    # 1. 전체 텍스트 토크나이징 (max_length로 패딩하여 모든 시퀀스 길이 통일)
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=config['data']['max_length'],
        padding="max_length",  # 모든 배치를 동일한 길이로 고정 (에러 방지 핵심)
        return_tensors=None
    )

    labels = [] # Initialize labels list
    for i, (prompt, response) in enumerate(zip(prompts, responses)):
        # 2. Prompt 길이 계산을 위해 별도 토크나이징 (add_special_tokens=False로 정확한 프롬프트 길이 측정)
        prompt_text = f"{prompt}\n"
        prompt_tokenized = tokenizer(
            prompt_text,
            add_special_tokens=True,
            truncation=True,
            max_length=config['data']['max_length'],
            return_tensors=None
        )
        prompt_len = len(prompt_tokenized['input_ids'])

        # 3. input_ids를 복사하여 labels 생성
        input_ids = tokenized['input_ids'][i]
        label = list(input_ids)

        # 4. Prompt 부분 마스킹
        mask_len = min(prompt_len, len(label))
        for j in range(mask_len):
            label[j] = '''이 부분을 채워주세요''' # 왜 특정 숫자로 이 부분을 설정해야하는지 생각해봅니다.

        # 5. 패딩 토큰 마스킹
        for j in range(len(label)):
          if tokenized['attention_mask'][i][j] == 0:
            label[j] = '''이 부분을 채워주세요''' # 왜 특정 숫자로 이 부분을 설정해야하는지 생각해봅니다.

        labels.append(label)

    tokenized["labels"] = labels

    return tokenized

# 데이터 토크나이징
print("\n데이터 토크나이징 중...")
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)
test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

# 정답 정보 저장 (평가용) - 이미 위에서 정의됨
# test_answers는 전처리 섹션에서 이미 정의되었습니다

print(f"토크나이징 완료!")
print(f"Train 데이터 shape: {len(train_dataset)}")
print(f"Test 데이터 shape: {len(test_dataset)}")

## 4. 모델 및 토크나이저 초기화

In [ ]:
# Base 모델 로드 (파인튜닝 전 성능 측정용)
print("Base 모델 로드 중...")
base_model = AutoModelForCausalLM.from_pretrained(
    config['model']['name_or_path'],
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32
)

print(f"모델 로드 완료!")
print(f"모델 파라미터 수: {sum(p.numel() for p in base_model.parameters()):,}")
print(f"학습 가능한 파라미터 수: {sum(p.numel() for p in base_model.parameters() if p.requires_grad):,}")


In [ ]:
# LoRA 설정
lora_config = LoraConfig(
    r='''이 부분을 채워주세요'''
    lora_alpha='''이 부분을 채워주세요'''
    target_modules='''이 부분을 채워주세요'''
    lora_dropout='''이 부분을 채워주세요'''
    bias=config'''이 부분을 채워주세요'''
    task_type=TaskType.CAUSAL_LM,
)

print("LoRA 설정:")
print(f"  - Rank (r): {lora_config.r}")
print(f"  - Alpha: {lora_config.lora_alpha}")
print(f"  - Target modules: {lora_config.target_modules}")
print(f"  - Dropout: {lora_config.lora_dropout}")
print(f"  - Bias: {lora_config.bias}")

# LoRA를 적용한 모델 생성
print("\nLoRA 모델 생성 중...")
model = get_peft_model('''이 부분을 채워주세요''')

# 학습 가능한 파라미터 확인
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n파라미터 통계:")
print(f"  - 전체 파라미터: {total_params:,}")
print(f"  - 학습 가능한 파라미터: {trainable_params:,}")
print(f"  - 학습 가능한 파라미터 비율: {100 * trainable_params / total_params:.2f}%")
print(f"  - 파라미터 절감: {total_params - trainable_params:,} ({100 * (1 - trainable_params / total_params):.2f}%)")

# 모델 구조 출력
model.print_trainable_parameters()


## 5. Base Model 평가 (파인튜닝 전)

In [ ]:
# 모델 평가 함수 (BLEU Score 계산)
def evaluate_model(model, tokenizer, test_processed, test_answers, config):
    """모델을 평가하고 BLEU 점수 반환"""
    model.eval()
    generated_translations = []
    reference_translations = []

    print("평가 중...")
    for i, item in enumerate(tqdm(test_processed)):
        prompt = item['prompt']

        # Prompt 토크나이징
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=config['data']['max_length'])
        if torch.cuda.is_available():
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # Generation
        '''이 부분을 채워주세요''':
            outputs = model.generate(
                **inputs,
                max_new_tokens=config['generation']['max_new_tokens'],
                do_sample=config['generation']['do_sample'],
                temperature=config['generation']['temperature'],
                num_beams=config['generation'].get('num_beams', 1), # num_beams 추가
                early_stopping=config['generation'].get('early_stopping', False), # early_stopping 추가
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # 디코딩
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # prompt 부분 제거
        # 모델이 생성하는 텍스트는 prompt + response 형태이므로, response 부분만 추출
        if prompt in generated_text:
            response_start_index = generated_text.find(prompt) + len(prompt)
            if response_start_index < len(generated_text):
                predicted_answer = generated_text[response_start_index:].strip()
            else:
                predicted_answer = ""
        else:
            # prompt가 생성된 텍스트에 포함되지 않는 경우 (이상 케이스), 전체를 예측으로 간주
            predicted_answer = generated_text.strip()

        # BLEU 점수 계산을 위해 토큰화하여 저장'''
        #if i <3:
        #  print(f"\n Model Generation: {generated_text}")
        #  print(f"\n Predicted Answer: {predicted_answer}\n Reference Answer: {test_answers[i]}\n=====\n")
        generated_translations.append(word_tokenize(predicted_answer.lower()))
        reference_translations.append([word_tokenize(test_answers[i].lower())])

    if len(generated_translations) > 0 and len(reference_translations) > 0:
        bleu_score = corpus_bleu('''이 부분을 채워주세요''') # bleu함수의 정의와 corpus_bleu 함수에 대해 생각해봅니다.
    else:
        bleu_score = 0.0

    return bleu_score, [" ".join(t) for t in generated_translations]

In [ ]:
# Base 모델 평가
print("Base 모델 평가 중...")
base_bleu_score, base_predictions = evaluate_model(
    '''이 부분을 채워주세요'''
)

print("\n" + "=" * 60)
print("Base Model 성능 (파인튜닝 전)")
print("=" * 60)
print(f"BLEU Score: {base_bleu_score:.4f}")
print("=" * 60)

## 6. LoRA Fine-tuning

In [ ]:
# TrainingArguments 설정
training_args = TrainingArguments(
    output_dir=config['training']['output_dir'],
    num_train_epochs=config['training']['num_train_epochs'],
    per_device_train_batch_size=config['training']['per_device_train_batch_size'],
    per_device_eval_batch_size=config['training']['per_device_eval_batch_size'],
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps'],
    learning_rate=config['training']['learning_rate'],
    logging_steps=config['training']['logging_steps'],
    eval_strategy="no",  # eval_dataset=None이므로 평가 비활성화
    save_strategy=config['training']['save_strategy'],
    load_best_model_at_end=False,  # eval_dataset이 없으므로 False
    save_total_limit=config['training']['save_total_limit'],
    bf16=config['training']['bf16'] if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else False,
    dataloader_num_workers=config['training']['dataloader_num_workers'],
    remove_unused_columns=config['training']['remove_unused_columns'],
    report_to=config['report_to'],
    seed=config['seed'],
)

# Data Collator 설정 (Language Modeling용)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM이므로 MLM은 False
)

# Trainer 초기화
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=None,  # Generation 평가는 별도로 수행
    tokenizer=tokenizer,
    data_collator=data_collator,
    # eval_dataset이 None이므로 EarlyStoppingCallback 제거
)

print("Trainer 설정 완료!")
print(f"출력 디렉토리: {training_args.output_dir}")
steps_per_epoch = len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
total_steps = steps_per_epoch * training_args.num_train_epochs
print(f"총 학습 스텝: {total_steps} (에포크당 {steps_per_epoch} 스텝)")

In [ ]:
# 학습 시작
print("=" * 60)
print("LoRA Fine-tuning 시작!")
print("=" * 60)

train_result = '''이 부분을 채워주세요''' # 학습을 실행하는 method입니다.

print("\n" + "=" * 60)
print("학습 완료!")
print("=" * 60)
print(f"총 학습 시간: {train_result.metrics['train_runtime']:.2f}초")
print(f"평균 학습 속도: {train_result.metrics['train_samples_per_second']:.2f} samples/sec")
print("=" * 60)


## 7. Fine-tuned Model 평가 (파인튜닝 후)

In [ ]:
# Fine-tuned 모델 평가
print("Fine-tuned 모델 평가 중...")
finetuned_bleu_score, finetuned_predictions = evaluate_model(
    '''이 부분을 채워주세요'''
)

print("\n" + "=" * 60)
print("Fine-tuned Model 성능 (파인튜닝 후)")
print("=" * 60)
print(f"BLEU Score: {finetuned_bleu_score:.4f}")
print("=" * 60)

## 8. 성능 비교: Base vs Fine-tuned

In [ ]:
# 성능 비교
print("=" * 60)
print("성능 비교: Base Model vs Fine-tuned Model")
print("=" * 60)

# 개선율 계산
improvement_bleu = '''이 부분을 채워주세요'''

print(f"\nBLEU Score:")
print(f"  Base Model:      {base_bleu_score:.4f}")
print(f"  Fine-tuned:      {finetuned_bleu_score:.4f}")
print(f"  개선율:          {improvement_bleu:+.2f}%")

print("\n" + "=" * 60)

In [ ]:
# 성능 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. BLEU Score 비교 바 차트
models = ['Base Model', 'Fine-tuned Model']
bleu_scores = [base_bleu_score, finetuned_bleu_score]
colors = ['#3498db', '#2ecc71']

bars = axes[0].bar(models, bleu_scores, color=colors, alpha=0.8, width=0.6)
axes[0].set_ylabel('BLEU Score')
axes[0].set_title('Base Model vs Fine-tuned Model')
axes[0].set_ylim([0, 1]) # BLEU score는 0에서 1 사이 값
axes[0].grid(axis='y', alpha=0.3)

# 값 표시
for bar, score in zip(bars, bleu_scores):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{score:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# 2. 개선율 차트
improvement_bleu = '''이 부분을 채워주세요'''
color = 'green' if improvement_bleu > 0 else 'red'

axes[1].barh(['BLEU Score'], [improvement_bleu], color=color, alpha=0.5)
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('Improvement (%)')
axes[1].set_title('Fine-tuned Model')
axes[1].grid(axis='x', alpha=0.3)

# 값 표시
axes[1].text(improvement_bleu + (1 if improvement_bleu > 0 else -1), 0, f'{improvement_bleu:+.2f}%',
            ha='left' if improvement_bleu > 0 else 'right', va='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('performance_comparison.png', dpi=150, bbox_inches='tight')
print("성능 비교 그래프 저장: performance_comparison.png")
plt.show()

## 9. 모델 저장 및 로드

In [ ]:
# 최종 모델 저장
final_model_path = config['training']['output_dir']
print(f"모델 저장 중: {final_model_path}")

# LoRA 가중치만 저장 (PEFT 방식)
trainer.save_model('''이 부분을 채워주세요''') # save_model 사용법에 대해 생각해봅니다.

# 토크나이저도 저장
tokenizer.save_pretrained('''이 부분을 채워주세요''') # save_pretrained 사용법에 대해 생각해봅니다.

print("✅ 모델 저장 완료!")
print(f"저장 위치: {final_model_path}")
print("\n저장된 파일:")
for file in os.listdir(final_model_path):
    if os.path.isfile(os.path.join(final_model_path, file)):
        size = os.path.getsize(os.path.join(final_model_path, file)) / (1024 * 1024)  # MB
        print(f"  - {file} ({size:.2f} MB)")


In [ ]:
# 저장된 모델 로드 테스트
print("저장된 모델 로드 테스트...")

# Base 모델과 LoRA 가중치를 함께 로드
loaded_model = PeftModel.from_pretrained(
    '''이 부분을 채워주세요'''
)

# 토크나이저 로드
loaded_tokenizer = AutoTokenizer.from_pretrained('''이 부분을 채워주세요''')

print("✅ 모델 로드 성공!")

# 간단한 테스트 (샘플 문제)
test_sample = test_processed[50]
test_prompt = test_sample['prompt']
test_response = test_sample['response'] # 'answer' 대신 'response' 사용

print(f"\n테스트 문제:")
print(f"Prompt: {test_prompt}")
print(f"정답: {test_response}")

inputs = loaded_tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=config['data']['max_length'])
if torch.cuda.is_available():
    inputs = {k: v.to(loaded_model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = loaded_model.generate(
        **inputs,
        max_new_tokens=config['generation']['max_new_tokens'],
        do_sample=config['generation']['do_sample'],
        temperature=config['generation']['temperature'],
        num_beams=config['generation'].get('num_beams', 1), # num_beams 추가
        early_stopping=config['generation'].get('early_stopping', False), # early_stopping 추가
        pad_token_id=loaded_tokenizer.pad_token_id,
        eos_token_id=loaded_tokenizer.eos_token_id
    )

generated_text = loaded_tokenizer.decode(outputs[0], skip_special_tokens=True)

# prompt 부분 제거
if test_prompt in generated_text:
    response_start_index = generated_text.find(test_prompt) + len(test_prompt)
    if response_start_index < len(generated_text):
        predicted_answer = generated_text[response_start_index:].strip()
    else:
        predicted_answer = ""
else:
    predicted_answer = generated_text.strip()

print(f"\n생성된 번역: {predicted_answer}")
print(f"정답 번역: {test_response}")

## 10. 학습 요약 및 정리

### 학습 포인트 정리

1. **LoRA의 효율성**
   - 전체 모델 파라미터 대비 매우 적은 파라미터만 학습
   - 메모리 효율적이고 빠른 학습 가능

2. **LoRA 하이퍼파라미터**
   - `r` (rank): LoRA의 rank, 낮을수록 파라미터 적음
   - `lora_alpha`: LoRA scaling factor
   - `target_modules`: LoRA를 적용할 레이어 선택

3. **HuggingFace Trainer**
   - 간편한 학습 파이프라인
   - 자동 평가, 체크포인트 저장, 로깅 등 지원

4. **성능 향상**
   - Task-specific 데이터로 파인튜닝 시 성능 개선 확인
   - Base model 대비 정확도 향상

### 추가 학습단계

- 다른 하이퍼파라미터로 실험해보기 (r, alpha, learning_rate 등)
- 다른 데이터셋으로 실험해보기
- 다른 모델로 실험해보기
- LoRA 가중치를 병합하여 단일 모델로 저장하기
